In [69]:
import pandas as pd
import psycopg2
import os

from dotenv import load_dotenv

load_dotenv("../.env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

In [70]:
conn = psycopg2.connect(
    host="awesome-hw.sdsc.edu",
    port=5432,
    dbname="nourish",
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

def query_db(query):
    """Query the nourish database and return data as a `pd.Dataframe`"""
    try:
        with conn.cursor() as cursor:
            cursor.execute(query)
            columns = [desc[0] for desc in cursor.description]
            rows = cursor.fetchall()

        return pd.DataFrame(rows, columns=columns)
    except Exception as e:
        conn.rollback()
        raise e

query_db("SELECT 1")
print(f"Connection successful")

Connection successful


In [71]:
def save_df_to_json(df: pd.DataFrame, filename: str):
    """Save a pandas DataFrame to a JSON file."""
    with open(f"../data/nodes/{filename}", "w") as f:
        df.to_json(f, index=False, orient="records", indent=2)

    print(f"Data saved to data/nodes/{filename}")

##### Entity 1: `State`

In [72]:
query = """
    SELECT
    1 as id,
    'CA' as code,
    'California' as name
"""

state_df = query_db(query)
save_df_to_json(state_df, "state.json")
print(f"Rows: {state_df.shape[0]}, Columns: {state_df.shape[1]}")
state_df.head()

Data saved to data/nodes/state.json
Rows: 1, Columns: 3


,id,code,name
0,1,CA,California


##### Entity 2: `County`

In [73]:
query= """
SELECT id, county as name
FROM county_neighborhoods
"""

county_df = query_db(query)
save_df_to_json(county_df, "county.json")
print(f"Rows: {county_df.shape[0]}, Columns: {county_df.shape[1]}")
county_df.head()

Data saved to data/nodes/county.json
Rows: 58, Columns: 2


,id,name
0,1,Alameda
1,2,Alpine
2,3,Amador
3,4,Butte
4,5,Calaveras


##### Entity 3: `City`

In [74]:
query = """
WITH cte AS (
    SELECT
        c.id,
        c.city,
        ST_Union(op.way) AS geom,  -- merge all polygons for that city
        MIN(ons.osm_id) AS osm_id  -- arbitrary representative ID
    FROM
        city_neighborhoods c
    LEFT JOIN osm_planet_socal_2025.osn_names ons
        ON c.city = ons.name
        AND ons.geom_type = 'polygon'
    LEFT JOIN osm_planet_socal_2025.planet_osm_polygon op
        ON ons.osm_id = op.osm_id
        AND ons.geom_type = 'polygon'
    WHERE
        op.osm_id < 0
        AND county = 'San Diego'
    GROUP BY
        c.city,
        c.id
)
SELECT
    id,
    city as name,
    ST_Transform(geom, 4326) AS geom,
    --st_aswkt(ST_Transform(geom, 4326)) AS geom_wkt,
    NULL AS geom_wkt,
    ST_Centroid(ST_Transform(geom, 4326)) AS centroid

FROM cte; 
"""


city_df = query_db(query)
save_df_to_json(city_df, "city.json")
print(f"Rows: {city_df.shape[0]}, Columns: {city_df.shape[1]}")
city_df.head()

Data saved to data/nodes/city.json
Rows: 52, Columns: 5


,id,name,geom,geom_wkt,centroid
0,8,Carlsbad,0103000020E6100000010000001D0300000EC40D53365A...,None,0101000020E610000049A785B9EB535DC041F8FAFE4F8F...
1,9,Chula Vista,0103000020E610000002000000170600009710BDD6EF47...,None,0101000020E61000003AB1DC6FEC405DC0D69967816650...
2,10,Coronado,0103000020E6100000020000007A0100000DF3D4D97F4E...,None,0101000020E61000000A7445D09F4A5DC054781F833252...
3,11,Del Mar,0103000020E610000001000000CD000000C7EDE1DC7051...,None,0101000020E610000023007032CF505DC09BE5A83B4C7B...
4,12,El Cajon,0103000020E61000000100000013090000DE2230D6B740...,None,0101000020E61000002CD85DA9783D5DC01DEE5A859D66...


##### Entity 4: `Community`

In [54]:
query = """
SELECT id, community as name
FROM community_neighborhoods
WHERE county = 'San Diego';
"""

community_df = query_db(query)
save_df_to_json(community_df, "community.json")
print(f"Rows: {community_df.shape[0]}, Columns: {community_df.shape[1]}")
community_df.head()

Data saved to data/nodes/community.json
Rows: 229, Columns: 2


,id,name
0,60,Midtown
1,55,Linda Vista
2,284,Eastlake Trails
3,285,Eastlake Vistas
4,283,Eastlake Land Swap


##### Entity 5: `Zipcode`

In [55]:
query = """
WITH san_diego_zipcodes AS (

SELECT DISTINCT CAST(unnest(zipcodes) AS TEXT) as zipcode
FROM city_neighborhoods
WHERE county = 'San Diego'

UNION

SELECT DISTINCT CAST(unnest(zipcodes) AS TEXT) as zipcode
FROM community_neighborhoods
WHERE county = 'San Diego'
)

SELECT

t.zipcode,
ST_Transform(s.geom, 4326) AS geom,
ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt,
ST_Centroid(ST_Transform(geom, 4326)) AS centroid

FROM
san_diego_zipcodes t
JOIN
test_zipcodes s ON t.zipcode = s.zip::text;
"""


zipcode_df = query_db(query)
zipcode_df["id"] = zipcode_df.index + 1
zipcode_df = zipcode_df[["id"]+[col for col in zipcode_df.columns if col != "id"]]
save_df_to_json(zipcode_df, "zipcode.json")
print(f"Rows: {zipcode_df.shape[0]}, Columns: {zipcode_df.shape[1]}")
zipcode_df.head()

Data saved to data/nodes/zipcode.json
Rows: 105, Columns: 5


,id,zipcode,geom,geom_wkt,centroid
0,1,91901,0106000020E61000000100000001030000000400000087...,MULTIPOLYGON(((6417254.00001338 1845596.000065...,0101000020E61000002744E82C37815841BC6005E39C9A...
1,2,91902,0106000020E610000001000000010300000007000000AA...,MULTIPOLYGON(((6324251.97571597 1831249.745293...,0101000020E61000005C8B6E39882258412990A28F05DA...
2,3,91905,0106000020E610000002000000010300000005000000F1...,MULTIPOLYGON(((6528173.3250903 1798504.4708891...,0101000020E6100000AFCF36D40DF758419F1E107E6914...
3,4,91906,0106000020E610000007000000010300000001000000E8...,MULTIPOLYGON(((6528506.99994789 1798956.999839...,0101000020E6100000DF44FF495EC35841D3BC20CCE6C9...
4,5,91910,0106000020E6100000020000000103000000010000000E...,MULTIPOLYGON(((6320499.99535905 1807998.290156...,0101000020E610000042238B4AA51258411E64444D2EA7...


##### Entity 6: `BusinessLocation`

In [66]:
query = """
SELECT    
    id,
    name,
    url,
    address,
    city,
    zip,
    latitude,
    longitude,
    blockgroup,
    categories,
    avg_rating,
    franchise,
    confidence,
    reasoning,
    ST_Transform(geom, 4326) AS geom,
    ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt
FROM ca_businesses_with_ai_franchise
"""

business_location_df = query_db(query)
business_location_df[business_location_df["zip"].isin(zipcode_df["zipcode"].values)]
avg_rating_fill = business_location_df["avg_rating"].mean()
business_location_df["avg_rating"] = business_location_df["avg_rating"].fillna(avg_rating_fill)
save_df_to_json(business_location_df, "business_location.json")
print(f"Rows: {business_location_df.shape[0]}, Columns: {business_location_df.shape[1]}")
business_location_df.head()

Data saved to data/nodes/business_location.json
Rows: 39593, Columns: 16


,id,name,url,address,city,zip,latitude,longitude,blockgroup,categories,avg_rating,franchise,confidence,reasoning,geom,geom_wkt
0,5,Internet Solutions For Less,https://www.google.com/maps/place//data=!4m2!3...,"Internet Solutions For Less, 733 Las Palmas Dr...",Vista,92081,33.1863539,-117.25029649999999,197021,"[Website designer, Design agency, Internet mar...",5,INDEPENDENT,0.85,The business name 'Internet Solutions For Less...,0101000020E610000068739CDB04505DC0B4FDD071DA97...,POINT(-117.2502965 33.1863539)
1,24,Wetzel's Pretzels,https://www.google.com/maps/place//data=!4m2!3...,"Wetzel's Pretzels, 869 W. Harbor Drive, #C2-F,...",San Diego,92101,32.708565199999995,-117.17027739999999,54023,[Pretzel store],3.9,FRANCHISE,0.95,Wetzel's Pretzels is a well-known chain specia...,0101000020E6100000DD0F2ED3E54A5DC0B68AB443B25A...,POINT(-117.1702774 32.7085652)
2,48,Del Mar Golf Center - Pelly's Mini Golf,https://www.google.com/maps/place//data=!4m2!3...,"Del Mar Golf Center - Pelly's Mini Golf, 15555...",Del Mar,92014,32.976022799999996,-117.2534354,83241,"[Golf driving range, Golf instructor, Miniatur...",4.4,INDEPENDENT,0.85,The business name 'Del Mar Golf Center - Pelly...,0101000020E610000084A91C4938505DC03E13AB50EE7C...,POINT(-117.2534354 32.9760228)
3,49,Brainy Actz Escape Rooms San Diego,https://www.google.com/maps/place//data=!4m2!3...,"Brainy Actz Escape Rooms San Diego, 10211 Paci...",San Diego,92121,32.9033799,-117.19029889999999,83462,"[Escape room center, Children's amusement cent...",4.3,FRANCHISE,0.85,The name 'Brainy Actz Escape Rooms' suggests a...,0101000020E6100000BEFD6FDB2D4C5DC08F2EDBF3A173...,POINT(-117.1902989 32.9033799)
4,50,Einstein Bros. Bagels,https://www.google.com/maps/place//data=!4m2!3...,"Einstein Bros. Bagels, 911 Lomas Santa Fe Dr, ...",Solana Beach,92075,32.9942243,-117.25523469999999,173061,"[Bagel shop, Bakery, Breakfast restaurant, Caf...",3.9,FRANCHISE,0.95,Einstein Bros. Bagels is a well-known chain wi...,0101000020E61000007A53ECC355505DC0BAB1EABD427F...,POINT(-117.2552347 32.9942243)


##### Entity 7: `Business`

In [57]:
business_df = business_location_df[["name"]].drop_duplicates().reset_index(drop=True)
business_df["id"] = business_df.index + 1
business_df = business_df[["id"]+[col for col in business_df.columns if col != "id"]]
save_df_to_json(business_df, "business.json")
print(f"Rows: {business_df.shape[0]}, Columns: {business_df.shape[1]}")
business_df.head()

Data saved to data/nodes/business.json
Rows: 32010, Columns: 2


,id,name
0,1,Internet Solutions For Less
1,2,Wetzel's Pretzels
2,3,Del Mar Golf Center - Pelly's Mini Golf
3,4,Brainy Actz Escape Rooms San Diego
4,5,Einstein Bros. Bagels


##### Entity 8: `BlockGroup`

In [75]:
query = """
SELECT
    sbg.ctblockgroup as id,
    bd.std_geography_id AS geo_id,
    sbg.ctblockgroup,

    -- INCOME
    imp.AVGDI_CY as average_income,
    imp.MEDDI_CY as median_income,
    imp.di0_cy    AS income_under_15000,
    imp.di15_cy   AS income_15000_24999,
    imp.di25_cy   AS income_25000_34999,
    imp.di35_cy   AS income_35000_49999,
    imp.di50_cy   AS income_50000_74999,
    imp.di75_cy   AS income_75000_99999,
    imp.di100_cy  AS income_100000_149999,
    imp.di150_cy  AS income_150000_199999,
    imp.di200_cy  AS income_200000_plus,

    -- POPULATION
    imp.TOTPOP_CY as total_population,

    -- AGE
    (imp.male0 + imp.male5 + imp.fem0 + imp.fem5) AS population_0_9,
    (imp.male10 + imp.male15 + imp.fem10 + imp.fem15) AS population_10_19,
    (imp.male20 + imp.male25 + imp.fem20 + imp.fem25) AS population_20_29,
    (imp.male30 + imp.male35 + imp.fem30 + imp.fem35) AS population_30_39,
    (imp.male40 + imp.male45 + imp.fem40 + imp.fem45) AS population_40_49,
    (imp.male50 + imp.male55 + imp.fem50 + imp.fem55) AS population_50_59,
    (imp.male60 + imp.male65 + imp.fem60 + imp.fem65) AS population_60_69,
    (imp.male70 + imp.male75 + imp.fem70 + imp.fem75) AS population_70_79,
    (imp.male80 + imp.male85 + imp.fem80 + imp.fem85) AS population_80_plus,

    -- MALE
    (imp.male0  + imp.male5)  AS male_population_0_9,
    (imp.male10 + imp.male15) AS male_population_10_19,
    (imp.male20 + imp.male25) AS male_population_20_29,
    (imp.male30 + imp.male35) AS male_population_30_39,
    (imp.male40 + imp.male45) AS male_population_40_49,
    (imp.male50 + imp.male55) AS male_population_50_59,
    (imp.male60 + imp.male65) AS male_population_60_69,
    (imp.male70 + imp.male75) AS male_population_70_79,
    (imp.male80 + imp.male85) AS male_population_80_plus,
    (imp.male0  + imp.male5  +
     imp.male10 + imp.male15 +
     imp.male20 + imp.male25 +
     imp.male30 + imp.male35 +
     imp.male40 + imp.male45 +
     imp.male50 + imp.male55 +
     imp.male60 + imp.male65 +
     imp.male70 + imp.male75 +
     imp.male80 + imp.male85) AS male_population_total,

    -- FEMALE
    (imp.fem0  + imp.fem5)  AS female_population_0_9,
    (imp.fem10 + imp.fem15) AS female_population_10_19,
    (imp.fem20 + imp.fem25) AS female_population_20_29,
    (imp.fem30 + imp.fem35) AS female_population_30_39,
    (imp.fem40 + imp.fem45) AS female_population_40_49,
    (imp.fem50 + imp.fem55) AS female_population_50_59,
    (imp.fem60 + imp.fem65) AS female_population_60_69,
    (imp.fem70 + imp.fem75) AS female_population_70_79,
    (imp.fem80 + imp.fem85) AS female_population_80_plus,
    (imp.fem0  + imp.fem5  +
     imp.fem10 + imp.fem15 +
     imp.fem20 + imp.fem25 +
     imp.fem30 + imp.fem35 +
     imp.fem40 + imp.fem45 +
     imp.fem50 + imp.fem55 +
     imp.fem60 + imp.fem65 +
     imp.fem70 + imp.fem75 +
     imp.fem80 + imp.fem85) AS female_population_total,

    -- BUSINESS COUNTS
    bd.s01_bus  AS businesses_total_sic,
    bd.s02_bus  AS businesses_agriculture_mining_sic,
    bd.s08_bus  AS businesses_wholesale_trade_sic,
    bd.s09_bus  AS businesses_retail_trade_sic,
    bd.s12_bus  AS businesses_food_stores_sic,
    bd.s13_bus  AS businesses_auto_gas_sic,
    bd.s16_bus  AS businesses_eating_drinking_sic,
    bd.s17_bus  AS businesses_misc_retail_sic,
    bd.s18_bus  AS businesses_finance_insurance_realestate_sic,
    bd.s19_bus  AS businesses_banks_sic,
    bd.s20_bus  AS businesses_securities_sic,
    bd.s21_bus  AS businesses_insurance_sic,
    bd.s22_bus  AS businesses_real_estate_sic,
    bd.s23_bus  AS businesses_services_sic,
    bd.s24_bus  AS businesses_hotels_sic,
    bd.s25_bus  AS businesses_auto_services_sic,
    bd.s26_bus  AS businesses_amusements_sic,
    bd.s27_bus  AS businesses_health_services_sic,
    bd.s28_bus  AS businesses_legal_services_sic,
    bd.s29_bus  AS businesses_education_sic,
    bd.s30_bus  AS businesses_other_services_sic,

    bd.n01_bus  AS businesses_total_naics,
    bd.n02_bus  AS businesses_agriculture_naics,
    bd.n07_bus  AS businesses_wholesale_trade_naics,
    bd.n08_bus  AS businesses_retail_trade_naics,
    bd.n12_bus  AS businesses_building_materials_naics,
    bd.n13_bus  AS businesses_food_beverage_stores_naics,
    bd.n14_bus  AS businesses_health_personal_care_naics,
    bd.n15_bus  AS businesses_gas_stations_naics,
    bd.n27_bus  AS businesses_real_estate_leasing_naics,
    bd.n35_bus  AS businesses_accommodation_food_naics,
    bd.n36_bus  AS businesses_accommodation_naics,
    bd.n37_bus  AS businesses_food_services_naics,

    -- BUSINESS SALES
    bd.s01_sales AS sales_total_sic,
    bd.s02_sales AS sales_agriculture_mining_sic,
    bd.s08_sales AS sales_wholesale_trade_sic,
    bd.s09_sales AS sales_retail_trade_sic,
    bd.s12_sales AS sales_food_stores_sic,
    bd.s13_sales AS sales_auto_gas_sic,
    bd.s16_sales AS sales_eating_drinking_sic,
    bd.s17_sales AS sales_misc_retail_sic,
    bd.s18_sales AS sales_finance_insurance_realestate_sic,
    bd.s19_sales AS sales_banks_sic,
    bd.s20_sales AS sales_securities_brokers_sic,
    bd.s21_sales AS sales_insurance_sic,
    bd.s22_sales AS sales_real_estate_sic,
    bd.s23_sales AS sales_services_sic,
    bd.s24_sales AS sales_hotels_sic,
    bd.s25_sales AS sales_auto_services_sic,
    bd.s26_sales AS sales_amusement_sic,
    bd.s27_sales AS sales_health_services_sic,
    bd.s28_sales AS sales_legal_services_sic,
    bd.s29_sales AS sales_education_sic,
    bd.s30_sales AS sales_other_services_sic,

    bd.n01_sales AS sales_total_naics,
    bd.n02_sales AS sales_agriculture_naics,
    bd.n07_sales AS sales_wholesale_trade_naics,
    bd.n08_sales AS sales_retail_trade_naics,
    bd.n12_sales AS sales_building_materials_naics,
    bd.n13_sales AS sales_food_beverage_stores_naics,
    bd.n14_sales AS sales_health_personal_care_naics,
    bd.n15_sales AS sales_gas_stations_naics,
    bd.n27_sales AS sales_real_estate_leasing_naics,
    bd.n35_sales AS sales_accommodation_food_naics,
    bd.n36_sales AS sales_accommodation_naics,
    bd.n37_sales AS sales_food_services_naics,

    ST_Transform(sbg.geom, 4326) AS geom,
    ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt,
    ST_Centroid(ST_Transform(geom, 4326)) AS centroid
FROM sandag_layer_census_block_groups sbg
LEFT JOIN bgs_sd_imp imp
    ON CAST(sbg.ctblockgroup AS TEXT) = CAST(CONCAT(LTRIM(imp.tractce, '0'), imp.blkgrpce) AS TEXT)
LEFT JOIN esri_business_data bd
    ON TRIM(LEADING '0' FROM SUBSTR(CAST(bd.std_geography_id AS TEXT), 5)) = CAST(sbg.ctblockgroup AS TEXT)
LEFT JOIN esri_consumer_spending_cols cs
    ON TRIM(LEADING '0' FROM SUBSTR(CAST(cs.std_geography_id AS TEXT), 5)) = CAST(sbg.ctblockgroup AS TEXT)
ORDER BY sbg.ctblockgroup ASC;
 
"""

block_group_df = query_db(query)
save_df_to_json(block_group_df, "block_group.json")
print(f"Rows: {block_group_df.shape[0]}, Columns: {block_group_df.shape[1]}")
block_group_df.head() 

Data saved to data/nodes/block_group.json
Rows: 2085, Columns: 113


,id,geo_id,ctblockgroup,average_income,median_income,income_under_15000,income_15000_24999,income_25000_34999,income_35000_49999,income_50000_74999,...,sales_food_beverage_stores_naics,sales_health_personal_care_naics,sales_gas_stations_naics,sales_real_estate_leasing_naics,sales_accommodation_food_naics,sales_accommodation_naics,sales_food_services_naics,geom,geom_wkt,centroid
0,1001,60730001001,1001,151149.0,126590.0,23.0,5.0,12.0,31.0,53.0,...,0,0,0,0,2642,2473,169,0106000020E610000001000000010300000001000000B8...,MULTIPOLYGON(((-117.188567830914 32.7591470904...,0101000020E6100000F9DD3870B24B5DC0799CEE388760...
1,1002,60730001002,1002,154312.0,136914.0,14.0,6.0,31.0,54.0,77.0,...,677,0,0,400,20786,8158,12628,0106000020E61000000100000001030000000100000031...,MULTIPOLYGON(((-117.187773001776 32.7572359994...,0101000020E6100000AFC67FB20C4C5DC02FD966214060...
2,2011,60730002011,2011,110484.0,102474.0,50.0,31.0,7.0,23.0,51.0,...,315,0,0,1998,2688,1608,1080,0106000020E61000000100000001030000000100000088...,MULTIPOLYGON(((-117.169610001679 32.7578239986...,0101000020E61000003299120A344B5DC0EC97B3AF9460...
3,2012,60730002012,2012,90034.0,70032.0,89.0,62.0,20.0,55.0,119.0,...,272,497,0,16269,3819,0,3819,0106000020E61000000100000001030000000100000062...,MULTIPOLYGON(((-117.172342000794 32.7556769991...,0101000020E6100000779362E9F44A5DC00471F9E05B60...
4,2021,60730002021,2021,87748.0,50577.0,97.0,96.0,46.0,111.0,92.0,...,814,0,5281,2048,14428,0,14428,0106000020E6100000010000000103000000010000008D...,MULTIPOLYGON(((-117.172285001534 32.7489329987...,0101000020E610000037A66881594B5DC022ECE625385F...


In [86]:
# show rows with duplicated ctblockgrou
block_group_df[block_group_df.duplicated(subset=['ctblockgroup'], keep=False)]

,id,geo_id,ctblockgroup,average_income,median_income,income_under_15000,income_15000_24999,income_25000_34999,income_35000_49999,income_50000_74999,...,sales_food_beverage_stores_naics,sales_health_personal_care_naics,sales_gas_stations_naics,sales_real_estate_leasing_naics,sales_accommodation_food_naics,sales_accommodation_naics,sales_food_services_naics,geom,geom_wkt,centroid
862,103001,60730103001,103001,85620.0,70478.0,22.0,24.0,27.0,38.0,52.0,...,0,0,0,0,0,0,0,0106000020E6100000010000000103000000010000004D...,MULTIPOLYGON(((-117.117939000487 32.5765619990...,0101000020E6100000A7E6E68475475DC0A5F52B7F7349...
863,103001,60730103001,103001,102604.0,83620.0,0.0,0.0,8.0,19.0,81.0,...,0,0,0,0,0,0,0,0106000020E6100000010000000103000000010000004D...,MULTIPOLYGON(((-117.117939000487 32.5765619990...,0101000020E6100000A7E6E68475475DC0A5F52B7F7349...
867,104011,60730104011,104011,24078.0,18731.0,57.0,112.0,42.0,16.0,6.0,...,814,0,3169,1515,2371,0,2371,0106000020E6100000010000000103000000010000005C...,MULTIPOLYGON(((-117.109113001378 32.5822759987...,0101000020E610000083C4E186DC465DC0280346C1294A...
868,104011,60730104011,104011,66280.0,55383.0,42.0,83.0,102.0,102.0,168.0,...,814,0,3169,1515,2371,0,2371,0106000020E6100000010000000103000000010000005C...,MULTIPOLYGON(((-117.109113001378 32.5822759987...,0101000020E610000083C4E186DC465DC0280346C1294A...
869,104021,60730104021,104021,54985.0,47108.0,53.0,70.0,99.0,117.0,181.0,...,4514,0,0,1006,2246,0,2246,0106000020E6100000010000000103000000010000007C...,MULTIPOLYGON(((-117.109382001197 32.5765479989...,0101000020E6100000FA6295EEC2465DC0F74B8F9D8F49...
870,104021,60730104021,104021,55230.0,50342.0,77.0,34.0,14.0,63.0,89.0,...,4514,0,0,1006,2246,0,2246,0106000020E6100000010000000103000000010000007C...,MULTIPOLYGON(((-117.109382001197 32.5765479989...,0101000020E6100000FA6295EEC2465DC0F74B8F9D8F49...
871,104022,60730104022,104022,47693.0,37361.0,67.0,51.0,29.0,52.0,47.0,...,0,0,0,4239,0,0,0,0106000020E61000000100000001030000000100000038...,MULTIPOLYGON(((-117.10819700119 32.57286999861...,0101000020E61000009C09D8CEE9465DC094C3850A1449...
872,104022,60730104022,104022,63758.0,51147.0,8.0,32.0,64.0,90.0,90.0,...,0,0,0,4239,0,0,0,0106000020E61000000100000001030000000100000038...,MULTIPOLYGON(((-117.10819700119 32.57286999861...,0101000020E61000009C09D8CEE9465DC094C3850A1449...
873,104023,60730104023,104023,52943.0,43887.0,134.0,77.0,101.0,167.0,191.0,...,543,0,0,1212,0,0,0,0106000020E61000000100000001030000000100000047...,MULTIPOLYGON(((-117.105021000706 32.5728609995...,0101000020E610000019D89AA59F465DC08A7DEDFB1549...
874,104023,60730104023,104023,53997.0,38712.0,67.0,152.0,97.0,72.0,75.0,...,543,0,0,1212,0,0,0,0106000020E61000000100000001030000000100000047...,MULTIPOLYGON(((-117.105021000706 32.5728609995...,0101000020E610000019D89AA59F465DC08A7DEDFB1549...


In [79]:
for c in block_group_df.columns:
    print(c)

id
geo_id
ctblockgroup
average_income
median_income
income_under_15000
income_15000_24999
income_25000_34999
income_35000_49999
income_50000_74999
income_75000_99999
income_100000_149999
income_150000_199999
income_200000_plus
total_population
population_0_9
population_10_19
population_20_29
population_30_39
population_40_49
population_50_59
population_60_69
population_70_79
population_80_plus
male_population_0_9
male_population_10_19
male_population_20_29
male_population_30_39
male_population_40_49
male_population_50_59
male_population_60_69
male_population_70_79
male_population_80_plus
male_population_total
female_population_0_9
female_population_10_19
female_population_20_29
female_population_30_39
female_population_40_49
female_population_50_59
female_population_60_69
female_population_70_79
female_population_80_plus
female_population_total
businesses_total_sic
businesses_agriculture_mining_sic
businesses_wholesale_trade_sic
businesses_retail_trade_sic
businesses_food_stores_sic
bu

##### Entity 9: `Zone Location`

In [59]:
query = """
SELECT
id,
zone_name,
imp_date,
ordnum,
shape_length,
shape_area,
legend,
ST_Transform(geom, 4326) AS geom,
ST_AsText(ST_Transform(geom, 4326)) AS geom_ewkt,
ST_Centroid(ST_Transform(geom, 4326)) AS centroid

FROM sandag_layer_zoning_base_sd_new
"""

zone_location_df = query_db(query)
save_df_to_json(zone_location_df, "zone_location.json")
print(f"Rows: {zone_location_df.shape[0]}, Columns: {zone_location_df.shape[1]}")
zone_location_df.head()

Data saved to data/nodes/zone_location.json
Rows: 3677, Columns: 10


,id,zone_name,imp_date,ordnum,shape_length,shape_area,legend,geom,geom_ewkt,centroid
0,1,AG-1-1,1141084800000,R-301263,281.423966,2.601150e+03,"Agricultural-General zone, use package 1, deve...",0106000020E6100000010000000103000000010000000E...,MULTIPOLYGON(((-117.12982648288 33.04642430605...,0101000020E610000058C500454C485DC0EA352115F485...
1,6,AG-1-1,1141084800000,R-301263,3197.909381,2.041669e+05,"Agricultural-General zone, use package 1, deve...",0106000020E61000000100000001030000000100000085...,MULTIPOLYGON(((-117.037822960772 33.0743520276...,0101000020E61000002A608A6D56425DC0F1B74447B289...
2,7,AG-1-1,1141084800000,R-301263,85544.431621,6.369870e+07,"Agricultural-General zone, use package 1, deve...",0106000020E61000000100000001030000000100000040...,MULTIPOLYGON(((-117.106843080255 33.0731224473...,0101000020E610000008FC9AE8AC465DC0D497FCB2CE87...
3,8,AG-1-1,1141084800000,R-301263,17855.440197,7.212937e+06,"Agricultural-General zone, use package 1, deve...",0106000020E610000001000000010300000001000000C8...,MULTIPOLYGON(((-116.972290881461 33.0752706838...,0101000020E6100000312BACEBAC3D5DC09146E002718A...
4,9,AG-1-1,1141084800000,R-301263,168.046707,1.087063e+03,"Agricultural-General zone, use package 1, deve...",0106000020E6100000010000000103000000010000000B...,MULTIPOLYGON(((-116.905763399499 33.0866176989...,0101000020E6100000EA51C073F9395DC06CA3C469168B...


##### Entity 10: `Zone Type`

In [61]:
query = """
SELECT
DISTINCT(zone_name) as name,
legend
FROM sandag_layer_zoning_base_sd_new
"""

zone_type_df = query_db(query)
zone_type_df["id"] = zone_type_df.index + 1
zone_type_df = zone_type_df[["id"]+[col for col in zone_type_df.columns if col != "id"]]
save_df_to_json(zone_type_df, "zone_type.json")
print(f"Rows: {zone_type_df.shape[0]}, Columns: {zone_type_df.shape[1]}")
zone_type_df.head()

Data saved to data/nodes/zone_type.json
Rows: 184, Columns: 3


,id,name,legend
0,1,OF-1-1,Open Space-Floodplain zone (floodplain areas)
1,2,CC-2-5,Other or Undefined Zone
2,3,CC-5-1,Commercial-Community zones (special areas)
3,4,CUPD-CT-2-3,Central Urbanized Planned District
4,5,LJPD-6A,La Jolla Planned District
